In [3]:
# -*- coding: utf-8 -*-
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd


# ========= 使用者參數 =========
P   = 10
BC  = "PBC"
chi = 40

# J, D, L 清單
Jstr = [f"Jdis{str(i).zfill(3)}" for i in range(10, 201, 10)]   # Jdis010..Jdis200
Dstr = [f"Dim{str(i).zfill(3)}" for i in range(0,1,10)]              # 只有 Dim000

Jstr = [f"Jdis{str(i).zfill(3)}" for i in range(120, 121, 10)]   # Jdis010..Jdis200
Dstr = [f"Dim{str(i).zfill(3)}" for i in range(1,51,1)]              # 只有 Dim000

# L: 8~64(步進8), 96~256(步進32), 384, 512
Lnums = list(range(8, 33, 8)) + [48, 64, 96] + list(range(128, 223, 32)) + [256, 384, 512]
# Lnums = [64, 128, 256, 384, 512]

Lnums  = [L for L in Lnums]

# 若你真的只想要 L=512，就改成 Lnums = [512]
# Lnums = []
Lstr  = [f"L{L}" for L in Lnums]

print(f"Pdis:{P}, BC:{BC}, chi:{chi}")
# path1=r"E:\Dropbox\metadataOutput0326"

# ========= 根目錄偵測 =========
candidates = [
    # "/ceph/work/NTHU-qubit/LYT/tSDRG_random",  # dicos
    # "/home/aronton/tSDRG_random/tSDRG/Main_15/data_random"              # scopion
    "/home/aronton/tSDRG_random/tSDRG/Main_15/metadata"
    # "./metadata"                                       # fallback: 目前目錄
    # "E:\Dropbox\dicos0202",
    # r"E:\Dropbox\metadataOutput0326\metadata"
    # r"D:\Users\Dropbox\dicos0206\metadata"
    # r"D:\Users\Dropbox\scopion\metadata"
]
root = next((Path(p) for p in candidates if Path(p).is_dir()), Path("."))
# root = Path(r"D:\Users\Dropbox\dicos\metadata")
base_metadata = root  # 你的原始程式看起來是讀 metadata 底下的東西
# base_metadata = root / "metadataOutput"
# ========= 小工具 =========
def read_meta_count(fp: Path) -> int:
    """
    讀 '..._meta' 檔案的第2行，抓逗號後的整數。
    若格式不同，會嘗試抓該行中的第一個整數；失敗則回 0。
    """
    try:
        lines = fp.read_text(encoding="utf-8", errors="ignore").splitlines()
        if len(lines) > 1:
            parts = lines[1].split(",")
            if len(parts) > 1:
                return int(parts[1])
            # 後備：抓第2行中的第一個整數
            m = re.search(r"-?\d+", lines[1])
            if m:
                return int(m.group(0))
        return 0
    except Exception:
        return 0
def read_dis_count(fp):

    try:
        lines = fp.read_text(encoding="utf-8", errors="ignore").splitlines()
        if lines[-1] == "\n":
            print(lines[-1])
        # 找第一個能轉 float 的位置
        for i, line in enumerate(lines):
            try:
                float(line.strip())
                return len(lines) - i
            except ValueError:
                continue

        return len(lines) - 1
    except Exception:
        return 0


def read_collect_count(fp: Path) -> int:
    """
    'collect' 模式：回傳資料行數（扣掉表頭 1 行），最少 0。
    """
    try:
        n = len(fp.read_text(encoding="utf-8", errors="ignore").splitlines())
        return max(0, n - 1)
    except Exception:
        return 0

def find_seed_for_L(base_dir: Path, L: str) -> int:
    """
    'seed' 模式：從 10000, 9000, ..., 1000 依序找第一個存在的 seed 資料夾。
    目錄格式：metadata/{BC}/{J}/{D}/{L}_P{P}_m{chi}_{seed}
    找不到回傳 0。
    """
    for seed in range(10000, 0, -1000):
        p = base_dir / f"{L}_P{P}_m{chi}_{seed}"
        if p.exists():
            return seed
    return 0

# ========= 主流程 =========

arr_OBC = {"corr":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)
           ,"gap":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)
           ,"gap2":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)}
arr_PBC = {"ZL":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)
           ,"corr":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)
           ,"gap":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)
           ,"gap2":np.zeros((len(Lstr), len(Jstr), len(Dstr)), dtype=int)}

for name, base_path, mode in [
    ("averaged", base_metadata, "meta"),   # 你原本只跑 meta；要跑別的改這裡
    # 例如：("collected", base_metadata, "collect"),
    #       ("seeded",    base_metadata, "seed"),
]:    
    BC  = "OBC"
    print(f"\n\n{name} data under: {base_path}/{BC}\n")
    # for J in Jstr:
    for d,D in enumerate(Dstr):
        for j,J in enumerate(Jstr):
        
        # for d,D in enumerate(Dstr):
            out_parts = []
            gap_list = []
            gap2_list = []
            ZL_list = []
            corr_list = []
            for l,L in enumerate(Lstr):
                # 資料夾與檔名
                if mode == "seed":
                    seed = find_seed_for_L(base_path / BC / J / D, L)
                    out_parts.append(f"{L}:{seed:>5}")
                    continue



                if d == 0 and j == 0 and l == 0:
                    print(f"{J}  -> {L:}  corr,gap ,gap2")

                folder  = f"{L}_P{P}_m{chi}" if mode == "collect" else f"{L}_P{P}_m{chi}_dis"
                L_1 = L.replace("L","")
                if BC == "PBC":
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)//2}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    ZLfname   = f"ZL_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                else:
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)-1}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                corrfpath   = base_path / BC / J / D / folder / corrfname
                gapfpath   = base_path / BC / J / D / folder / gapfname
                gap2fpath   = base_path / BC / J / D / folder / gap2fname

                # if not corrfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")
                #     # continue

                # if not gapfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not gap2fpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not ZLfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                if mode == "collect":
                    cnt = read_collect_count(fpath)
                elif mode == "meta":
                    corrCnt = read_dis_count(corrfpath)
                    gapCnt = read_dis_count(gapfpath)
                    gap2Cnt = read_dis_count(gap2fpath)
                else:
                    corrCnt = 0
                    gapCnt = 0
                    gap2Cnt = 0
                Lnum = int((L.replace("L","")))
                # out_parts.append(f"{L}:{2*cnt//Lnum:>5}")
                corr_list.append(corrCnt)
                gap_list.append(gapCnt)
                gap2_list.append(gap2Cnt)
                arr_OBC["corr"][l][j][d] = corrCnt
                arr_OBC["gap"][l][j][d] = gapCnt
                arr_OBC["gap2"][l][j][d] = gap2Cnt
                out_parts.append(f"{L}:{corrCnt//1000:>4},{gapCnt//1000:>4},{gap2Cnt//1000:>4}k")
                # out_parts.append(f"{gapCnt//1000:>5}")
                # out_parts.append(f"{gap2Cnt//1000:>5}")
                # out_parts.append(f"{ZLCnt//1000:>5}")

            # 每個 (J,D) 輸出一行
            # if sum(cnt_list) != 0:
            print(f"{D} -> " + "; ".join(out_parts))
    
    
    
    
    
    BC  = "PBC"
    print(f"\n\n{name} data under: {base_path}/{BC}\n")
    # for J in Jstr:
    for d,D in enumerate(Dstr):
        for j,J in enumerate(Jstr):
    # for j,J in enumerate(Jstr):
        
    #     for d,D in enumerate(Dstr):
            out_parts = []
            gap_list = []
            gap2_list = []
            ZL_list = []
            corr_list = []
            for l,L in enumerate(Lstr):
                # 資料夾與檔名
                if mode == "seed":
                    seed = find_seed_for_L(base_path / BC / J / D, L)
                    out_parts.append(f"{L}:{seed:>5}")
                    continue



                if d == 0 and j == 0 and l == 0:
                    print(f"{J}  -> {L:}  corr,gap ,gap2 ,ZL  ")

                folder  = f"{L}_P{P}_m{chi}" if mode == "collect" else f"{L}_P{P}_m{chi}_dis"
                L_1 = L.replace("L","")
                if BC == "PBC":
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)//2}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    ZLfname   = f"ZL_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                else:
                    fname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)-1}.txt"
                corrfpath   = base_path / BC / J / D / folder / corrfname
                gapfpath   = base_path / BC / J / D / folder / gapfname
                gap2fpath   = base_path / BC / J / D / folder / gap2fname
                ZLfpath   = base_path / BC / J / D / folder / ZLfname

                # if not corrfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")
                #     # continue

                # if not gapfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not gap2fpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not ZLfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")


                if mode == "collect":
                    cnt = read_collect_count(fpath)
                elif mode == "meta":
                    corrCnt = read_dis_count(corrfpath)
                    corrCnt = 2*(corrCnt//Lnums[l])
                    gapCnt = read_dis_count(gapfpath)
                    gap2Cnt = read_dis_count(gap2fpath)
                    ZLCnt = read_dis_count(ZLfpath)
                else:
                    corrCnt = 0
                    gapCnt = 0
                    gap2Cnt = 0
                    ZLCnt = 0
                Lnum = int((L.replace("L","")))
                # out_parts.append(f"{L}:{2*cnt//Lnum:>5}")
                corr_list.append(corrCnt)
                gap_list.append(gapCnt)
                gap2_list.append(gap2Cnt)
                ZL_list.append(ZLCnt)
                arr_PBC["corr"][l][j][d] = corrCnt
                arr_PBC["gap"][l][j][d] = gapCnt
                arr_PBC["gap2"][l][j][d] = gap2Cnt
                arr_PBC["ZL"][l][j][d] = ZLCnt
                out_parts.append(f"{L}:{corrCnt//1000:>4},{gapCnt//1000:>4},{gap2Cnt//1000:>4},{ZLCnt//1000:>4}k")
                # out_parts.append(f"{gapCnt//1000:>5}")
                # out_parts.append(f"{gap2Cnt//1000:>5}")
                # out_parts.append(f"{ZLCnt//1000:>5}")

            # 每個 (J,D) 輸出一行
            # if sum(cnt_list) != 0:
            print(f"{D} -> " + "; ".join(out_parts))

Pdis:10, BC:PBC, chi:40


averaged data under: /home/aronton/tSDRG_random/tSDRG/Main_15/metadata/OBC

Jdis120  -> L8  corr,gap ,gap2
Dim001 -> L8:   0,   0,   0k; L16:   0,   0,   0k; L24:   0,   0,   0k; L32:   0,   0,   0k; L48:   0,   0,   0k; L64:   0,   0,   0k; L96:   0,   0,   0k; L128:   0,   0,   0k; L160:   0,   0,   0k; L192:   0,   0,   0k; L256:   0,   0,   0k; L384:   0,   0,   0k; L512:   0,   0,   0k
Dim002 -> L8:   0,   0,   0k; L16:   0,   0,   0k; L24:   0,   0,   0k; L32:   0,   0,   0k; L48:   0,   0,   0k; L64:   0,   0,   0k; L96:   0,   0,   0k; L128:   0,   0,   0k; L160:   0,   0,   0k; L192:   0,   0,   0k; L256:   0,   0,   0k; L384:   0,   0,   0k; L512:   0,   0,   0k
Dim003 -> L8:   0,   0,   0k; L16:   0,   0,   0k; L24:   0,   0,   0k; L32:   0,   0,   0k; L48:   0,   0,   0k; L64:   0,   0,   0k; L96:   0,   0,   0k; L128:   0,   0,   0k; L160:   0,   0,   0k; L192:   0,   0,   0k; L256:   0,   0,   0k; L384:   0,   0,   0k; L512:   0,   0,   0k
Dim004

In [2]:
import os
import math
import time
import timeit
import numpy as np
import sys
import tarfile
import datetime
import multiprocessing
import scriptCreator
from pathlib import Path
import shutil


dicosPath = "/dicos_ui_home/aronton/sharedfs/work/NTHU-qubit/LYT/tSDRG_random"
scopionPath = "/home/aronton/tSDRG_random"

if os.path.isdir(dicosPath):
    tSDRG_path = dicosPath
    group_path = dicosPath
    
if os.path.isdir(scopionPath):
    tSDRG_path = scopionPath
    group_path = scopionPath
    
print(tSDRG_path)

/home/aronton/tSDRG_random


## R

In [12]:
import os
import pathlib


J_i, J_f, J_d = 10, 201, 10
J_list = [f"Jdis{str(i).zfill(3)}" for i in range(J_i, J_f, J_d)]
J_num = [int(s.replace('Jdis', '')) / 100.0 for s in J_list]

L_i, L_f, L_d = 64, 513, 64
L_list = [f"L{num-1}" for num in range(L_i, L_f, L_d)]
L_num = [i-1 for i in range(L_i, L_f, L_d)]

D = "Dim000"
BC = "OBC"
P = 10
B = 40
print(f"BC: {BC}, Pdis: {P}, Bond: {B}")
for j,J in enumerate(J_list):
    if not os.path.exists(f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{J}/{D}"):
        continue
    else:
        print("R"+J.replace("Jdis",""))
    output = []
    for l,L in enumerate(L_num):
        # L=L-1
        physlist = {"ZL": 0, "corr": 0, "energy": 0}
        
        for seed in range(30000, 0, -1000):
            path = f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{J}/{D}/L{L}_P{P}_m{B}_{seed}"
            if not os.path.exists(path):
                continue
        
            ZLpath     = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
            Corrpath   = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
            energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"
        
            found_any = False
            if os.path.exists(ZLpath):
                physlist["ZL"] = seed // 1000
                found_any = True
            if os.path.exists(Corrpath):
                physlist["corr"] = seed // 1000
                found_any = True
            if os.path.exists(energypath):              # ✅ 修正這行
                physlist["energy"] = seed // 1000
                found_any = True
        
            if found_any:
                break

        # for seed in range(30000,0,-1000):
        #     path = f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{J}/Dim000/L{L}_P{P}_m{B}_{seed}"
        #     if os.path.exists(path):
        #         # print("path:", path)
        #         ZLpath = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
        #         Corrpath = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
        #         energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"
        #         if os.path.exists(ZLpath):
        #             physlist["ZL"]  = seed//1000
        #         if os.path.exists(Corrpath):
        #             physlist["corr"]  = seed//1000
        #         if os.path.exists(energypath):
        #             physlist["energy"]  = seed//1000
        #         break
        # print("path:", path)
        output.append(
            f"L{L} -> {physlist.get('ZL')}, {physlist.get('corr')}, {physlist.get('energy')}(k)"
        )
    print("_".join(output))


BC: OBC, Pdis: 10, Bond: 40
R010
L63 -> 10, 10, 10(k)_L127 -> 10, 10, 10(k)_L191 -> 10, 10, 10(k)_L255 -> 3, 3, 3(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R020
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 0, 0, 0(k)_L255 -> 0, 0, 0(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R030
L63 -> 10, 10, 10(k)_L127 -> 5, 5, 5(k)_L191 -> 5, 5, 5(k)_L255 -> 9, 9, 9(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R040
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 0, 0, 0(k)_L255 -> 0, 0, 0(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R050
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 10, 10, 10(k)_L255 -> 10, 10, 10(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R060
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 0, 0, 0(k)_L255 -> 0, 0, 0(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
R070
L63 -> 0, 0, 

## D

In [5]:
import os
import pathlib

Jstr = "Jdis020"

D_i, D_f, D_d = 0, 51, 1
D_list = [f"Dim{str(i).zfill(3)}" for i in range(D_i, D_f, D_d)]
D_num = [int(s.replace('Dim', '')) / 100.0 for s in D_list]

L_i, L_f, L_d = 64, 513, 64
L_list = [f"L{num-1}" for num in range(L_i, L_f, L_d)]
L_num = [i-1 for i in range(L_i, L_f, L_d)]

BC = "OBC"
P = 10
B = 40
print(f"BC: {BC}, Pdis: {P}, Bond: {B}")
for d,D in enumerate(D_list):
    if not os.path.exists(f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{Jstr}/{D}"):
        continue
    else:
        print("D"+D.replace("Dim",""))
    output = []
    for l,L in enumerate(L_num):
        # L=L-1
        physlist = {"ZL": 0, "corr": 0, "energy": 0}
        
        for seed in range(30000, 0, -1000):
            path = f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{Jstr}/{D}/L{L}_P{P}_m{B}_{seed}"
            if not os.path.exists(path):
                continue
        
            ZLpath     = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
            Corrpath   = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
            energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"
        
            found_any = False
            if os.path.exists(ZLpath):
                physlist["ZL"] = seed // 1000
                found_any = True
            if os.path.exists(Corrpath):
                physlist["corr"] = seed // 1000
                found_any = True
            if os.path.exists(energypath):              # ✅ 修正這行
                physlist["energy"] = seed // 1000
                found_any = True
        
            if found_any:
                break

        # for seed in range(30000,0,-1000):
        #     path = f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{J}/Dim000/L{L}_P{P}_m{B}_{seed}"
        #     if os.path.exists(path):
        #         # print("path:", path)
        #         ZLpath = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
        #         Corrpath = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
        #         energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"
        #         if os.path.exists(ZLpath):
        #             physlist["ZL"]  = seed//1000
        #         if os.path.exists(Corrpath):
        #             physlist["corr"]  = seed//1000
        #         if os.path.exists(energypath):
        #             physlist["energy"]  = seed//1000
        #         break
        # print("path:", path)
        output.append(
            f"L{L} -> {physlist.get('ZL')}, {physlist.get('corr')}, {physlist.get('energy')}(k)"
        )
    print("_".join(output))

BC: OBC, Pdis: 10, Bond: 40
D000
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 0, 0, 0(k)_L255 -> 0, 0, 0(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
D010
L63 -> 0, 0, 0(k)_L127 -> 0, 0, 0(k)_L191 -> 0, 0, 0(k)_L255 -> 5, 5, 5(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
D020
L63 -> 0, 0, 0(k)_L127 -> 5, 5, 5(k)_L191 -> 0, 0, 0(k)_L255 -> 5, 5, 5(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
D030
L63 -> 0, 0, 0(k)_L127 -> 5, 5, 5(k)_L191 -> 0, 0, 0(k)_L255 -> 4, 4, 4(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
D040
L63 -> 0, 0, 0(k)_L127 -> 5, 5, 5(k)_L191 -> 0, 0, 0(k)_L255 -> 3, 3, 3(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)
D050
L63 -> 0, 0, 0(k)_L127 -> 5, 5, 5(k)_L191 -> 0, 0, 0(k)_L255 -> 2, 2, 2(k)_L319 -> 0, 0, 0(k)_L383 -> 0, 0, 0(k)_L447 -> 0, 0, 0(k)_L511 -> 0, 0, 0(k)


In [8]:
import os
import pathlib

dirName = "S15_restored_random"
dirName = "data_random"


J_i, J_f, J_d = 40, 41, 10
J_list = [f"Jdis{str(i).zfill(3)}" for i in range(J_i, J_f, J_d)]
J_num = [int(s.replace('Jdis', '')) / 100.0 for s in J_list]


D_i, D_f, D_d = 5, 51, 1
D_list = [f"Dim{str(i).zfill(3)}" for i in range(D_i, D_f, D_d)]
D_num = [int(s.replace('Dim', '')) / 100.0 for s in D_list]

L_i, L_f, L_d = 64, 513, 32
L_list = [f"L{num}" for num in range(L_i, L_f, L_d)]
L_num = [i for i in range(L_i, L_f, L_d)]

BC = "PBC"
P = 10
B = 40
print(f"BC: {BC}, Pdis: {P}, Bond: {B}")
J = "Jdis040"

for d,D in enumerate(D_list):
    if not os.path.exists(f"{tSDRG_path}/tSDRG/Main_15/{dirName}/{BC}/{J}/{D}"):
        continue
    else:
        # print("R"+J.replace("Jdis",""))
        print(D)
    output = []
    for l,L in enumerate(L_num):
        # L=L-1
        physlist = {"ZL": 0, "corr": 0, "energy": 0}
        
        for seed in range(30000, 0, -1000):
            path = f"{tSDRG_path}/tSDRG/Main_15/{dirName}/{BC}/{J}/{D}/L{L}_P{P}_m{B}_{seed}"
            if not os.path.exists(path):
                continue
        
            ZLpath     = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
            Corrpath   = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
            energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"

            # ZLpath     = f"{path}/ZL.txt"
            # Corrpath   = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
            # energypath = f"{path}/energy.csv"

            found_any = False
            if os.path.exists(ZLpath):
                physlist["ZL"] = seed // 1000
                found_any = True
            if os.path.exists(Corrpath):
                physlist["corr"] = seed // 1000
                found_any = True
            if os.path.exists(energypath):              # ✅ 修正這行
                physlist["energy"] = seed // 1000
                found_any = True
        
            if found_any:
                break

        # for seed in range(30000,0,-1000):
        #     path = f"{tSDRG_path}/tSDRG/Main_15/data_random/{BC}/{J}/Dim000/L{L}_P{P}_m{B}_{seed}"
        #     if os.path.exists(path):
        #         # print("path:", path)
        #         ZLpath = f"{path}/L{L}_P{P}_m{B}_{seed}_ZL.txt"
        #         Corrpath = f"{path}/L{L}_P{P}_m{B}_{seed}_corr1.txt"
        #         energypath = f"{path}/L{L}_P{P}_m{B}_{seed}_energy.txt"
        #         if os.path.exists(ZLpath):
        #             physlist["ZL"]  = seed//1000
        #         if os.path.exists(Corrpath):
        #             physlist["corr"]  = seed//1000
        #         if os.path.exists(energypath):
        #             physlist["energy"]  = seed//1000
        #         break
        # print("path:", path)
        if (physlist["energy"] + physlist["ZL"] + physlist["corr"]) != 0:
            output.append(
                f"L{L} -> {physlist.get('ZL')}, {physlist.get('corr')}, {physlist.get('energy')}(k)"
            )
    print("_".join(output))


BC: PBC, Pdis: 10, Bond: 40
Dim005
L128 -> 4, 4, 4(k)_L256 -> 4, 4, 4(k)_L512 -> 3, 3, 3(k)
Dim010
L128 -> 5, 5, 5(k)_L256 -> 5, 5, 5(k)_L512 -> 5, 5, 5(k)
Dim015
L128 -> 4, 4, 4(k)_L256 -> 4, 4, 4(k)_L512 -> 1, 1, 1(k)
Dim020
L128 -> 3, 3, 3(k)_L512 -> 5, 5, 5(k)
Dim025
L128 -> 4, 4, 4(k)_L256 -> 4, 4, 4(k)_L512 -> 1, 1, 1(k)
Dim030
L128 -> 5, 5, 5(k)_L256 -> 5, 5, 5(k)_L512 -> 5, 5, 5(k)
Dim040

Dim042

Dim044
L64 -> 1, 1, 1(k)_L128 -> 3, 3, 3(k)
Dim046
L64 -> 1, 1, 1(k)_L128 -> 10, 10, 10(k)
Dim048

Dim050
L64 -> 7, 7, 7(k)_L128 -> 10, 10, 10(k)


In [1]:
# -*- coding: utf-8 -*-
import os
import re
from pathlib import Path

# ========= 使用者參數 =========
P   = 10
BC  = "OBC"
chi = 40

# J, D, L 清單
Jstr = [f"Jdis{str(i).zfill(3)}" for i in range(20, 21, 10)]   # Jdis010..Jdis200
Dstr = [f"Dim{str(i).zfill(3)}" for i in range(0,51,1)]              # 只有 Dim000

# L: 8~64(步進8), 96~256(步進32), 384, 512
Lnums = list(range(8, 33, 8)) + [48, 64, 96] + list(range(128, 223, 32)) + [256, 384, 512]
# 若你真的只想要 L=512，就改成 Lnums = [512]
# Lnums = []
Lstr  = [f"L{L-1}" for L in Lnums]

print(f"Pdis:{P}, BC:{BC}, chi:{chi}")

# ========= 根目錄偵測 =========
candidates = [
    # "/ceph/work/NTHU-qubit/LYT/tSDRG_random",  # dicos
    "/home/aronton/tSDRG_random/tSDRG/Main_15/metadata"              # scopion
    # "./metadata"                                       # fallback: 目前目錄
    # r"E:\Dropbox\dicos0206\metadata"
]
root = next((Path(p) for p in candidates if Path(p).is_dir()), Path("."))
# root = Path(r"D:\Users\Dropbox\dicos\metadata")
base_metadata = root  # 你的原始程式看起來是讀 metadata 底下的東西
print("base_metadata", base_metadata)
# base_metadata = root / "metadataOutput"
# ========= 小工具 =========
def read_meta_count(fp: Path) -> int:
    """
    讀 '..._meta' 檔案的第2行，抓逗號後的整數。
    若格式不同，會嘗試抓該行中的第一個整數；失敗則回 0。
    """
    try:
        lines = fp.read_text(encoding="utf-8", errors="ignore").splitlines()
        if len(lines) > 1:
            parts = lines[1].split(",")
            if len(parts) > 1:
                return int(parts[1])
            # 後備：抓第2行中的第一個整數
            m = re.search(r"-?\d+", lines[1])
            if m:
                return int(m.group(0))
        return 0
    except Exception:
        return 0
def read_dis_count(fp):

    try:
        lines = fp.read_text(encoding="utf-8", errors="ignore").splitlines()
        if lines[-1] == "\n":
            print(lines[-1])
        # 找第一個能轉 float 的位置
        for i, line in enumerate(lines):
            try:
                float(line.strip())
                return len(lines) - i
            except ValueError:
                continue

        return len(lines) - 1
    except Exception:
        return 0


def read_collect_count(fp: Path) -> int:
    """
    'collect' 模式：回傳資料行數（扣掉表頭 1 行），最少 0。
    """
    try:
        n = len(fp.read_text(encoding="utf-8", errors="ignore").splitlines())
        return max(0, n - 1)
    except Exception:
        return 0

def find_seed_for_L(base_dir: Path, L: str) -> int:
    """
    'seed' 模式：從 10000, 9000, ..., 1000 依序找第一個存在的 seed 資料夾。
    目錄格式：metadata/{BC}/{J}/{D}/{L}_P{P}_m{chi}_{seed}
    找不到回傳 0。
    """
    for seed in range(10000, 0, -1000):
        p = base_dir / f"{L}_P{P}_m{chi}_{seed}"
        if p.exists():
            return seed
    return 0

# ========= 主流程 =========
for name, base_path, mode in [
    ("averaged", base_metadata, "meta"),   # 你原本只跑 meta；要跑別的改這裡
    # 例如：("collected", base_metadata, "collect"),
    #       ("seeded",    base_metadata, "seed"),
]:
    
    print(f"\n\n{name} data under: {base_path}/{BC}\n")
    # for J in Jstr:
    for j,J in enumerate(Jstr):
        
        for d,D in enumerate(Dstr):
            out_parts = []
            gap_list = []
            gap2_list = []
            ZL_list = []
            corr_list = []
            for l,L in enumerate(Lstr):
                # 資料夾與檔名
                if mode == "seed":
                    seed = find_seed_for_L(base_path / BC / J / D, L)
                    out_parts.append(f"{L}:{seed:>5}")
                    continue



                if d == 0 and j == 0 and l == 0:
                    print(f"{D}  -> {L:}  corr,gap ,gap2")

                folder  = f"{L}_P{P}_m{chi}" if mode == "collect" else f"{L}_P{P}_m{chi}_dis"
                L_1 = L.replace("L","")
                if BC == "PBC":
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)//2}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                else:
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)-1}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                corrfpath   = base_path / BC / J / D / folder / corrfname
                gapfpath   = base_path / BC / J / D / folder / gapfname
                gap2fpath   = base_path / BC / J / D / folder / gap2fname

                # if not corrfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")
                #     # continue

                # if not gapfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not gap2fpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not ZLfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")


                if mode == "collect":
                    cnt = read_collect_count(fpath)
                elif mode == "meta":
                    corrCnt = read_dis_count(corrfpath)
                    gapCnt = read_dis_count(gapfpath)
                    gap2Cnt = read_dis_count(gap2fpath)
                else:
                    corrCnt = 0
                    gapCnt = 0
                    gap2Cnt = 0
                Lnum = int((L.replace("L","")))
                # out_parts.append(f"{L}:{2*cnt//Lnum:>5}")
                corr_list.append(corrCnt)
                gap_list.append(gapCnt)
                gap2_list.append(gap2Cnt)

                out_parts.append(f"{L}:{corrCnt//1000:>4},{gapCnt//1000:>4},{gap2Cnt//1000:>4}k")
                # out_parts.append(f"{gapCnt//1000:>5}")
                # out_parts.append(f"{gap2Cnt//1000:>5}")
                # out_parts.append(f"{ZLCnt//1000:>5}")

            # 每個 (J,D) 輸出一行
            # if sum(cnt_list) != 0:
            print(f"{J} -> " + "; ".join(out_parts))
    
    
    
    
    print(f"\n\n{name} data under: {base_path}/{BC}\n")
    # for J in Jstr:
    for j,J in enumerate(Jstr):
        
        for d,D in enumerate(Dstr):
            out_parts = []
            gap_list = []
            gap2_list = []
            ZL_list = []
            corr_list = []
            for l,L in enumerate(Lstr):
                # 資料夾與檔名
                if mode == "seed":
                    seed = find_seed_for_L(base_path / BC / J / D, L)
                    out_parts.append(f"{L}:{seed:>5}")
                    continue



                if d == 0 and j == 0 and l == 0:
                    print(f"{D}  -> {L:}  corr,gap ,gap2 ,ZL  ")

                folder  = f"{L}_P{P}_m{chi}" if mode == "collect" else f"{L}_P{P}_m{chi}_dis"
                L_1 = L.replace("L","")
                if BC == "PBC":
                    corrfname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)//2}.txt"
                    gapfname   = f"gap_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    gap2fname   = f"gap2_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                    ZLfname   = f"ZL_dis_{L}_P{P}_m{chi}_{J}_{D}.txt"
                else:
                    fname   = f"corr1_dis_{L}_P{P}_m{chi}_{J}_{D}_dx={int(L_1)-1}.txt"
                corrfpath   = base_path / BC / J / D / folder / corrfname
                gapfpath   = base_path / BC / J / D / folder / gapfname
                gap2fpath   = base_path / BC / J / D / folder / gap2fname
                ZLfpath   = base_path / BC / J / D / folder / ZLfname
                # print(corrfpath)
                # if not corrfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")
                #     # continue

                # if not gapfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not gap2fpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")

                # if not ZLfpath.exists():
                #     out_parts.append(f"{L}:{0:>5}")


                if mode == "collect":
                    cnt = read_collect_count(fpath)
                elif mode == "meta":
                    corrCnt = read_dis_count(corrfpath)
                    gapCnt = read_dis_count(gapfpath)
                    gap2Cnt = read_dis_count(gap2fpath)
                    ZLCnt = read_dis_count(ZLfpath)
                else:
                    corrCnt = 0
                    gapCnt = 0
                    gap2Cnt = 0
                    ZLCnt = 0
                Lnum = int((L.replace("L","")))
                # out_parts.append(f"{L}:{2*cnt//Lnum:>5}")
                corr_list.append(corrCnt)
                gap_list.append(gapCnt)
                gap2_list.append(gap2Cnt)
                ZL_list.append(ZLCnt)

                out_parts.append(f"{L}:{2*(corrCnt//Lnums[l])//1000:>4},{gapCnt//1000:>4},{gap2Cnt//1000:>4},{ZLCnt//1000:>4}k")
                # if J == "Jdis120":
                #     print(corrfpath)
                #     print(corrCnt, ZLCnt)
                #     print(f"{L}:{2*(corrCnt//Lnums[l])//1000:>4},{gapCnt//1000:>4},{gap2Cnt//1000:>4},{ZLCnt//1000:>4}k")
                # out_parts.append(f"{gapCnt//1000:>5}")
                # out_parts.append(f"{gap2Cnt//1000:>5}")
                # out_parts.append(f"{ZLCnt//1000:>5}")

            # 每個 (J,D) 輸出一行
            # if sum(cnt_list) != 0:
            print(f"{J} -> " + "; ".join(out_parts))

Pdis:10, BC:OBC, chi:40
base_metadata /home/aronton/tSDRG_random/tSDRG/Main_15/metadata


averaged data under: /home/aronton/tSDRG_random/tSDRG/Main_15/metadata/OBC

Dim000  -> L7  corr,gap ,gap2
Jdis020 -> L7:   0,   0,   0k; L15:   0,   0,   0k; L23:   0,   0,   0k; L31:   0,   0,   0k; L47:   0,   0,   0k; L63:   0,   0,   0k; L95:   0,   0,   0k; L127:   0,   0,   0k; L159:   0,   0,   0k; L191:   0,   0,   0k; L255:   0,   0,   0k; L383:   0,   0,   0k; L511:   0,   0,   0k
Jdis020 -> L7:   0,   0,   0k; L15:   0,   0,   0k; L23:   0,   0,   0k; L31:   0,   0,   0k; L47:   0,   0,   0k; L63:   0,   0,   0k; L95:   0,   0,   0k; L127:   0,   0,   0k; L159:   0,   0,   0k; L191:   0,   0,   0k; L255:   0,   0,   0k; L383:   0,   0,   0k; L511:   0,   0,   0k
Jdis020 -> L7:   0,   0,   0k; L15:   0,   0,   0k; L23:   0,   0,   0k; L31:   0,   0,   0k; L47:   0,   0,   0k; L63:   0,   0,   0k; L95:   0,   0,   0k; L127:   0,   0,   0k; L159:   0,   0,   0k; L191:   0,   0,   0k; L255:

NameError: name 'ZLfname' is not defined